# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [4]:
import pandas as pd
from IPython.display import display

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

display(df.head())

# Question 1
low_claim_yes = df[(df["Total Claim Amount"] < 1000) & (df["Response"] == "Yes")]
display(low_claim_yes.head())
print(f"Shape: {low_claim_yes.shape}")

# Exercise two
yes_df = df[df["Response"] == "Yes"]

yes_summary = yes_df.groupby(["Policy Type", "Gender"]).agg(
    avg_monthly_premium=("Monthly Premium Auto", "mean"),
    avg_clv=("Customer Lifetime Value", "mean"),
    avg_total_claim=("Total Claim Amount", "mean"),
    median_total_claim=("Total Claim Amount", "median"),
    customers=("Response", "size")
).reset_index()

display(yes_summary)

# 3
customers_by_state = df.groupby("State").size().reset_index(name="customers")
customers_over_500 = customers_by_state[customers_by_state["customers"] > 500]
display(customers_over_500)

#  4
clv_stats = df.groupby(["Education", "Gender"])["Customer Lifetime Value"].agg(["min", "median", "max"]).reset_index()
display(clv_stats)

# B 5
df["Effective To Date"] = pd.to_datetime(df["Effective To Date"])
df["month"] = df["Effective To Date"].dt.month

policies_pivot = df.pivot_table(values="Policy", index="State", columns="month", aggfunc="count", fill_value=0)
display(policies_pivot)

# Bonus 6
state_month_policies = df.groupby(["State", "month"]).size().reset_index(name="policy_count")
top_3_states = df.groupby("State").size().sort_values(ascending=False).head(3).index
top_3_policies = state_month_policies[state_month_policies["State"].isin(top_3_states)]
top_3_pivot = top_3_policies.pivot_table(values="policy_count", index="State", columns="month", fill_value=0)
display(top_3_pivot)

# Bonus seven
response_by_channel = df.groupby("Sales Channel")["Response"].apply(lambda x: (x == "Yes").sum() / len(x) * 100).reset_index(name="response_rate_percent")
display(response_by_channel)

,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
8,8,FM55990,California,5989.773931,Yes,Premium,College,1/19/11,Employed,M,...,0.0,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN
15,15,CW49887,California,4626.801093,Yes,Basic,Master,1/16/11,Employed,F,...,0.0,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2/26/11,Employed,F,...,1.0,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2/28/11,Employed,F,...,0.0,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN


Shape: (1399, 26)


,Policy Type,Gender,avg_monthly_premium,avg_clv,avg_total_claim,median_total_claim,customers
0,Corporate Auto,F,94.301775,7712.628736,433.738499,420.041981,169
1,Corporate Auto,M,92.188312,7944.465414,408.582459,364.800000,154
2,Personal Auto,F,98.998148,8339.791842,452.965929,424.330166,540
3,Personal Auto,M,91.085821,7448.383281,457.010178,412.800000,536
4,Special Auto,F,92.314286,7691.584111,453.280164,420.041981,35
5,Special Auto,M,86.343750,8247.088702,429.527942,345.600000,32


,State,customers
0,Arizona,1937
1,California,3552
2,Nevada,993
3,Oregon,2909
4,Washington,888


,Education,Gender,min,median,max
0,Bachelor,F,1904.000852,5640.505303,73225.95652
1,Bachelor,M,1898.007675,5548.031892,67907.27050
2,College,F,1898.683686,5623.611187,61850.18803
3,College,M,1918.119700,6005.847375,61134.68307
4,Doctor,F,2395.570000,5332.462694,44856.11397
5,Doctor,M,2267.604038,5577.669457,32677.34284
6,High School or Below,F,2144.921535,6039.553187,55277.44589
7,High School or Below,M,1940.981221,6286.731006,83325.38119
8,Master,F,2417.777032,5729.855012,51016.06704
9,Master,M,2272.307310,5579.099207,50568.25912


/var/folders/8w/jbgrr0fd4jvbmhsmhc4cb5400000gn/T/ipykernel_35522/3984617959.py:37: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Effective To Date"] = pd.to_datetime(df["Effective To Date"])


month,1,2
State,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425


month,1,2
State,,
Arizona,1008.0,929.0
California,1918.0,1634.0
Oregon,1565.0,1344.0


,Sales Channel,response_rate_percent
0,Agent,18.005339
1,Branch,10.787558
2,Call Center,10.322279
3,Web,10.885609


In [ ]:
# bonus - Overall this suggests  customers who respond yes generally being of a higher value, especially in personal and corporate categories, where lifetime value is higher but claims do not seem to rise by the same amount. Personal looks like a fairly safe/good area to focus on, with solid average and median lifetime value and relatively steady claims, while corporate also performs well, particularly among females, even if there is slightly more variation. California and Oregon stand out mainly because of their scale, there are a lot of customers in these states so even small improvements in response or retention could have a noticeable overall impact on the business, which makes them good places to focus first. Education level also seems to play a role, with custs with masters and doctorates tending to show higher lifetime value and more long-term upside, which supports more tailored, value-focused messaging rather than across the board discounts. Sex/gender differences are present but not very strong, so they probably should not drive targeting decisions on their own. Overall the data suggests focusing on high-volume locations, personal and corporate customers, and higher value segments where returns look solid and claims risk appears broadly manageable, even allowing for some natural variation or fluctuations in the combineddata.
